# Emotion Classification: Inference
**Group 3, Section 7MA1**

This notebook loads the models we already trained in `Emotions_classification.ipynb`, so there is no training here. It is ready in about a minute.

Before using it, the last cell of the main notebook must have been run once. That cell saves the models to Google Drive in `MyDrive/NLP_Project_Group3/saved_models`.

A GPU is not needed. RoBERTa runs fine on CPU for single sentences.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, joblib, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SAVE_DIR = '/content/drive/MyDrive/NLP_Project_Group3/saved_models'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)

## 2. Load the saved models

In [ ]:
# labels and settings used during training
info = json.load(open(f'{SAVE_DIR}/info.json'))
EKMAN      = info['labels']        # ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']
THRESHOLD  = info['threshold']     # 0.5
MAX_LENGTH = info['max_length']    # 48 tokens

# Method 1: TF-IDF vectorizer + logistic regression
tfidf = joblib.load(f'{SAVE_DIR}/tfidf_logreg.joblib')
tf_vectorizer = tfidf['vectorizer']
tfidf_model   = tfidf['model']

# Method 2: fine-tuned RoBERTa
r_tokenizer   = AutoTokenizer.from_pretrained(f'{SAVE_DIR}/roberta_finetuned')
roberta_model = AutoModelForSequenceClassification.from_pretrained(f'{SAVE_DIR}/roberta_finetuned').to(device)
roberta_model.eval()

print("Loaded. Labels:", EKMAN)

## 3. Prediction function
Same logic as the main notebook: an emotion is predicted when its probability is above 0.5.

In [ ]:
@torch.no_grad()
def predict_emotions(text):
    if not isinstance(text, str) or text.strip() == "":
        print("Please type a non-empty English sentence.")
        return

    # Method 1: TF-IDF + LogReg (columns are in EKMAN order)
    tfidf_pred = tfidf_model.predict(tf_vectorizer.transform([text]))[0]
    tfidf_labels = [EKMAN[i] for i in range(7) if tfidf_pred[i] == 1]

    # Method 2: fine-tuned RoBERTa
    enc = r_tokenizer([text], truncation=True, max_length=MAX_LENGTH, return_tensors='pt').to(device)
    probs = torch.sigmoid(roberta_model(**enc).logits)[0].cpu().numpy()
    roberta_labels = [EKMAN[i] for i in range(7) if probs[i] > THRESHOLD]

    print(text)
    print("  TF-IDF :", tfidf_labels or "(no emotion above 0.5)")
    print("  RoBERTa:", roberta_labels or f"(no emotion above 0.5, most likely: {EKMAN[probs.argmax()]})")
    print("  RoBERTa probabilities:", {EKMAN[i]: round(float(probs[i]), 2) for i in range(7)})

## 4. Try it
Type a sentence in the box and run the cell.

In [ ]:
text = "I can't believe they cancelled the trip again."  #@param {type:"string"}
predict_emotions(text)

### Demo sentences

In [ ]:
examples = [
    "My hands are shaking as I open the results email.",
    "I finally got the internship, I can't stop smiling!",
    "Wow, thanks for nothing.",
    "",
]
for sentence in examples:
    predict_emotions(sentence)
    print()